# ============================================================
# Customer Analytics & Churn Prediction
# Notebook 07 - Prediction Pipeline
# ============================================================

In [56]:
import pandas as pd
import numpy as np

import joblib
from pathlib import Path

In [57]:
BASE_DIR = Path.cwd().parent

MODEL_DIR = BASE_DIR / "models"

DATA_DIR = BASE_DIR / "data"

PROCESSED_DATA = DATA_DIR / "processed" / "telco_customer_churn_clean.csv"

In [58]:
X_train = pd.read_csv(BASE_DIR / "data" / "processed" / "X_train.csv")
X_test = pd.read_csv(BASE_DIR / "data" / "processed" / "X_test.csv")

df = pd.concat([X_train, X_test], ignore_index=True)

df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,1,0,1,1,1.321816,1,2,1,2,2,2,2,0,0,2,0,1,0.981556,1.659900
1,1,0,0,0,-0.267410,0,1,0,0,0,2,2,0,0,0,0,2,-0.971546,-0.562252
2,0,0,1,0,1.444064,1,2,1,0,2,2,2,0,0,2,0,1,0.837066,1.756104
3,1,0,0,0,-1.204646,1,0,1,0,2,0,0,0,2,0,0,2,0.641092,-0.908326
4,0,0,1,0,0.669826,0,1,0,2,0,0,0,2,0,0,0,0,-0.808787,-0.101561


In [59]:
best_model = joblib.load(MODEL_DIR / "best_random_forest.pkl")

scaler = joblib.load(MODEL_DIR / "scaler.pkl")

print("Models loaded successfully.")

Models loaded successfully.


In [60]:
df.columns.tolist()

['gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'MonthlyCharges',
 'TotalCharges']

In [61]:
X = df.copy()

feature_columns = X.columns.tolist()

print(feature_columns)

['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']


In [62]:
def predict_customer(customer_data):

    customer = pd.DataFrame([customer_data])

    numerical_cols = [
        "tenure",
        "MonthlyCharges",
        "TotalCharges"
    ]

    customer[numerical_cols] = scaler.transform(
        customer[numerical_cols]
    )

    probability = best_model.predict_proba(customer)[0][1]

    prediction = best_model.predict(customer)[0]

    return prediction, probability

In [63]:
sample_customer_1 = X.iloc[0].to_dict()

prediction, probability = predict_customer(sample_customer_1)

print("Prediction:", prediction)
print("Probability:", probability)

Prediction: 0
Probability: 0.13668245545169289


In [64]:
sample_customer_2 = X.iloc[150].to_dict()

prediction, probability = predict_customer(sample_customer_2)

print("Prediction:", prediction)
print("Probability:", probability)

Prediction: 1
Probability: 0.591598265077296


In [65]:
def risk_level(probability):

    if probability < 0.30:
        return "Low Risk"

    elif probability < 0.70:
        return "Medium Risk"

    else:
        return "High Risk"

In [66]:
prediction, probability = predict_customer(sample_customer_2)

print("=" * 40)

print("Prediction")

print("=" * 40)

print("Customer Churn Prediction :",
      "Yes" if prediction == 1 else "No")

print(f"Churn Probability : {probability:.2%}")

print("Risk Level :", risk_level(probability))

Prediction
Customer Churn Prediction : Yes
Churn Probability : 59.16%
Risk Level : Medium Risk


In [67]:
results = []

for i in range(10):

    customer = X.iloc[i].to_dict()

    pred, prob = predict_customer(customer)

    results.append({
        "Customer": i,
        "Prediction": pred,
        "Probability": round(prob,4),
        "Risk": risk_level(prob)
    })

results_df = pd.DataFrame(results)

results_df

,Customer,Prediction,Probability,Risk
0,0,0,0.1367,Low Risk
1,1,0,0.3955,Medium Risk
2,2,0,0.2684,Low Risk
3,3,1,0.6189,Medium Risk
4,4,1,0.5791,Medium Risk
5,5,0,0.3196,Medium Risk
6,6,0,0.3832,Medium Risk
7,7,1,0.6670,Medium Risk
8,8,0,0.3752,Medium Risk
9,9,1,0.7611,High Risk


In [68]:
OUTPUT_DIR = BASE_DIR / "outputs"

OUTPUT_DIR.mkdir(exist_ok=True)

results_df.to_csv(
    OUTPUT_DIR / "sample_predictions.csv",
    index=False
)

print("Predictions saved.")

Predictions saved.


In [69]:
joblib.dump(
    feature_columns,
    MODEL_DIR / "feature_columns.pkl"
)

print("Feature names saved.")

Feature names saved.


In [70]:
print("="*50)

print("Prediction Pipeline Ready")

print("="*50)

print(f"Model Loaded        : {type(best_model).__name__}")

print(f"Features Used       : {len(feature_columns)}")

print(f"Scaler Loaded       : Yes")

print("Pipeline Status     : Ready for Deployment")

print("="*50)

Prediction Pipeline Ready
Model Loaded        : RandomForestClassifier
Features Used       : 19
Scaler Loaded       : Yes
Pipeline Status     : Ready for Deployment


In [71]:
print(df.columns)
print(df.head())
print(df.shape)

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges'],
      dtype='str')
   gender  SeniorCitizen  Partner  Dependents    tenure  PhoneService  \
0       1              0        1           1  1.321816             1   
1       1              0        0           0 -0.267410             0   
2       0              0        1           0  1.444064             1   
3       1              0        0           0 -1.204646             1   
4       0              0        1           0  0.669826             0   

   MultipleLines  InternetService  OnlineSecurity  OnlineBackup  \
0              2                1               2             2   
1              1                0               0             0   
2    